# S2 - Fundamentos PySpark: transformaciones, funciones, agrupaciones y evaluación perezosa

**Actividad:** construir el notebook `02_fundamentos_practica.ipynb` sobre el entorno `lambda26` (`uso-pyspark`), aplicando extracción, transformaciones, funciones, agrupaciones/agregaciones, RDD y verificando en cada paso el efecto de la evaluación perezosa — sobre el dataset real H&M (Kaggle).

**Propósito de la actividad:** dejar evidencia ejecutable de que dominas el ciclo completo de transformación distribuida en PySpark — DataFrame y RDD sobre la misma `SparkSession` — antes de avanzar a formatos analíticos particionados (S3) y ML distribuido (S4).

Guía completa: `docs/sesiones/S02_Fundamentos_PySpark_Transformaciones_Lazy_Evaluation.md`, sección 3 (pasos 3.1 a 3.11).

## 3.1 Descargar el dataset H&M y reanudar el entorno `lambda26`

**Producto del paso:** dataset H&M disponible en `pyspark/sesiones/s02-fundamentos/data/`, entorno `lambda26` funcionando.

El dataset ya está descargado en `data/` (`articles.csv`, `customers.csv`, `transactions.parquet`). Si el contenedor `lambda26` sigue corriendo desde S1, continúa directo en 3.2.

## 3.2 Crear el notebook y la `SparkSession`

**Producto del paso:** notebook con una `SparkSession` activa.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion2-fundamentos-spark")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/20 21:44:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


`spark.driver.memory` en `"4g"`: en modo `local[*]` el driver y el executor comparten un mismo proceso JVM, y sin fijar esto Spark usa el default de 1g — sin importar cuánta RAM tenga el servidor. Con varias lecturas/escrituras y agrupaciones encadenadas en un mismo notebook, 1g se queda corto y la JVM puede caerse (se ve como `ConnectionRefusedError` del lado de Python, aunque el servidor tenga memoria de sobra sin usar). Si el notebook se cae igual, reinicia el kernel y sube este valor (`"8g"`, `"16g"`, según la RAM disponible).

Declara la ruta del dataset como variable global, una sola vez — úsala en cada lectura del resto del notebook.

In [2]:
ORIGEN_DATOS = "/opt/s02-fundamentos/data"
ARTIFACTS = "/opt/s02-fundamentos/artifacts"

## 3.3 Cargar y explorar `articles.csv`

**Producto del paso:** DataFrame `df_articles` cargado (primera forma de lectura: `inferSchema`), con estructura, filas y estadísticas verificadas paso a paso.

Primero, la lectura:

In [3]:
df_articles = spark.read.csv(
    f"{ORIGEN_DATOS}/articles.csv",
    header=True,
    inferSchema=True,
)

`.show()` — visualiza filas en formato tabla:

In [4]:
df_articles.show(5, truncate=False)

+----------+------------+-----------------+---------------+-----------------+------------------+-----------------------+-------------------------+-----------------+-----------------+-------------------------+---------------------------+--------------------------+----------------------------+-------------+---------------+----------+----------------+--------------+----------------+----------+----------------------+----------------+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|article_id|product_code|prod_name        |product_type_no|product_type_name|product_group_name|graphical_appearance_no|graphical_appearance_name|colour_group_code|colour_group_name|perceived_colour_value_id|perceived_colour_value_name|perceived_colour_master_id|perceived_colour_master_name|department_no|

`.printSchema()` — muestra el esquema (nombres, tipos, nulabilidad):

In [5]:
df_articles.printSchema()

root
 |-- article_id: integer (nullable = true)
 |-- product_code: integer (nullable = true)
 |-- prod_name: string (nullable = true)
 |-- product_type_no: integer (nullable = true)
 |-- product_type_name: string (nullable = true)
 |-- product_group_name: string (nullable = true)
 |-- graphical_appearance_no: integer (nullable = true)
 |-- graphical_appearance_name: string (nullable = true)
 |-- colour_group_code: integer (nullable = true)
 |-- colour_group_name: string (nullable = true)
 |-- perceived_colour_value_id: integer (nullable = true)
 |-- perceived_colour_value_name: string (nullable = true)
 |-- perceived_colour_master_id: integer (nullable = true)
 |-- perceived_colour_master_name: string (nullable = true)
 |-- department_no: integer (nullable = true)
 |-- department_name: string (nullable = true)
 |-- index_code: string (nullable = true)
 |-- index_name: string (nullable = true)
 |-- index_group_no: integer (nullable = true)
 |-- index_group_name: string (nullable = true)

`.describe()` — resumen estadístico. Con las 25 columnas de `articles.csv`, ni siquiera `vertical=True` lo deja cómodo (125 líneas) — mejor selecciona antes un puñado de columnas representativas, mezclando nominales y numéricas:

In [6]:
df_articles.select(
    "prod_name", "product_group_name", "colour_group_name", "department_no", "section_no"
).describe().show()

26/08/20 21:44:28 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 3:>                                                          (0 + 9) / 9]

+-------+--------------------+------------------+-----------------+------------------+-----------------+
|summary|           prod_name|product_group_name|colour_group_name|     department_no|       section_no|
+-------+--------------------+------------------+-----------------+------------------+-----------------+
|  count|              105542|            105542|           105542|            105542|           105542|
|   mean|                NULL|              NULL|             NULL|  4532.77783252165|42.66421898391162|
| stddev|                NULL|              NULL|             NULL|2712.6920114304908|23.26010495890632|
|    min|& Denim Boyfriend...|       Accessories|            Beige|              1201|                2|
|    max|           Åsa Dress|           Unknown|  Yellowish Brown|              9989|               97|
+-------+--------------------+------------------+-----------------+------------------+-----------------+



Parámetros de `.show()`: `vertical=True` muestra cada fila como lista de campos, útil con muchas columnas.

In [7]:
df_articles.show(3, vertical=True)

-RECORD 0--------------------------------------------
 article_id                   | 108775015            
 product_code                 | 108775               
 prod_name                    | Strap top            
 product_type_no              | 253                  
 product_type_name            | Vest top             
 product_group_name           | Garment Upper body   
 graphical_appearance_no      | 1010016              
 graphical_appearance_name    | Solid                
 colour_group_code            | 9                    
 colour_group_name            | Black                
 perceived_colour_value_id    | 4                    
 perceived_colour_value_name  | Dark                 
 perceived_colour_master_id   | 5                    
 perceived_colour_master_name | Black                
 department_no                | 1676                 
 department_name              | Jersey Basic         
 index_code                   | A                    
 index_name                 

## 3.4 Cargar `customers.csv` con esquema explícito y explorar columnas

**Producto del paso:** DataFrame `df_customers` cargado con `StructType` (segunda forma de lectura), con sus columnas y su tamaño verificados.

Define el esquema, columna por columna:

In [8]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

schema_customers = StructType([
    StructField("customer_id", StringType(), nullable=True),
    StructField("FN", DoubleType(), nullable=True),
    StructField("Active", DoubleType(), nullable=True),
    StructField("club_member_status", StringType(), nullable=True),
    StructField("fashion_news_frequency", StringType(), nullable=True),
    StructField("age", IntegerType(), nullable=True),
    StructField("postal_code", StringType(), nullable=True),
])

Lee el CSV con ese esquema:

In [9]:
df_customers = spark.read.csv(
    f"{ORIGEN_DATOS}/customers.csv",
    header=True,
    schema=schema_customers,
)

Resumen estadístico:

In [10]:
df_customers.describe().show()

[Stage 7:==========>                                              (9 + 41) / 50]

+-------+--------------------+------+------+------------------+----------------------+------------------+--------------------+
|summary|         customer_id|    FN|Active|club_member_status|fashion_news_frequency|               age|         postal_code|
+-------+--------------------+------+------+------------------+----------------------+------------------+--------------------+
|  count|             1371980|476930|464404|           1365918|               1355971|           1356119|             1371980|
|   mean|                NULL|   1.0|   1.0|              NULL|                  NULL|   36.386964565794|                NULL|
| stddev|                NULL|   0.0|   0.0|              NULL|                  NULL|14.313627981628082|                NULL|
|    min|00000dbacae5abe5e...|   1.0|   1.0|            ACTIVE|               Monthly|                16|0000198d2c593b7d3...|
|    max|ffffd9ac14e899464...|   1.0|   1.0|        PRE-CREATE|             Regularly|                99|ffffe1

Nombres de columna:

In [11]:
print(df_customers.columns)

['customer_id', 'FN', 'Active', 'club_member_status', 'fashion_news_frequency', 'age', 'postal_code']


Cantidad de filas y de columnas:

In [12]:
num_rows, num_cols = df_customers.count(), len(df_customers.columns)
print(f"Filas: {num_rows}, Columnas: {num_cols}")

Filas: 1371980, Columnas: 7


**Muestra aleatoria** (`.sample()`): con ~1.37 millones de filas, trabajar con todo el dataset en una laptop se vuelve pesado — para explorar y validar la lógica alcanza con una muestra.

In [13]:
df_customers_muestra = df_customers.sample(
    withReplacement=False,  # sin reemplazo: cada fila se elige como máximo una vez
    fraction=0.01,          # ~1% del dataset
    seed=None,              # sin semilla fija: cada corrida da una muestra distinta
)

**Registros iniciales** (`.head()`):

In [14]:
df_customers_muestra.head(3)

[Row(customer_id='00008469a21b50b3d147c97135e25b4201a8c58997f78782a0cc706645e14493', FN=None, Active=None, club_member_status='ACTIVE', fashion_news_frequency='NONE', age=20, postal_code='2c29ae653a9282cce4151bd87643c907644e09541abc28ae87dea0d1f6603b1c'),
 Row(customer_id='0009f11f6ed21711acd1fcfe3d3cb3a1f542a95c50a897fa390bc2d60b4a11fa', FN=1.0, Active=1.0, club_member_status='ACTIVE', fashion_news_frequency='Regularly', age=27, postal_code='2c29ae653a9282cce4151bd87643c907644e09541abc28ae87dea0d1f6603b1c'),
 Row(customer_id='000a452812a2c2f7d6de7bb8971a6d51593018d5a3af8c2d161235ae2968f181', FN=None, Active=None, club_member_status='ACTIVE', fashion_news_frequency='NONE', age=19, postal_code='549ba569ea7206483088fbe5ae47d9d0cd3adeb60c372b1004cd2f86445c018f')]

**Registros finales** (`.tail()`):

In [15]:
df_customers_muestra.tail(3)

[Row(customer_id='fff472b503d848c0446b040b2475dccba2c1a9d1f0541bd1869e493bbfd0ea94', FN=1.0, Active=1.0, club_member_status='ACTIVE', fashion_news_frequency='Regularly', age=20, postal_code='19eeeec09e6be90551ff341246f3c4abaa90df3a3bce88fd0a49a2234e8d7fd3'),
 Row(customer_id='fff67e0ce3800aab23d16104d8ae13224f80053000dd6e79fb37672fc66b8c5d', FN=None, Active=None, club_member_status='ACTIVE', fashion_news_frequency='NONE', age=27, postal_code='396f2170e1924bdbafa13b3a406789ddd615ac07e482936eaad6661ad02b8546'),
 Row(customer_id='fff871bf24b40fd1290215414d760afaa69bb164d2b97000ef0030280674d0d9', FN=None, Active=None, club_member_status='ACTIVE', fashion_news_frequency='NONE', age=45, postal_code='1d7da83238816126546aee18740dec958ee1b82d93c278308868a575b4ebca83')]

**Guardar la muestra en CSV y Parquet:** adelanto de lo que S3 formaliza a fondo — por ahora, solo la mecánica básica de escribir un resultado a disco. Spark escribe una **carpeta** (un archivo por partición adentro), no un solo archivo; `mode("overwrite")` reemplaza la carpeta si ya existe.

In [16]:
df_customers_muestra.write.mode("overwrite").csv(f"{ARTIFACTS}/customers_muestra_csv", header=True)
df_customers_muestra.write.mode("overwrite").parquet(f"{ARTIFACTS}/customers_muestra_parquet")

26/08/20 21:44:32 WARN MemoryManager: Total allocation exceeds 95.00% (4,080,218,880 bytes) of heap memory
Scaling row group sizes to 98.06% for 31 writers
26/08/20 21:44:32 WARN MemoryManager: Total allocation exceeds 95.00% (4,080,218,880 bytes) of heap memory
Scaling row group sizes to 95.00% for 32 writers
26/08/20 21:44:32 WARN MemoryManager: Total allocation exceeds 95.00% (4,080,218,880 bytes) of heap memory
Scaling row group sizes to 92.12% for 33 writers
26/08/20 21:44:32 WARN MemoryManager: Total allocation exceeds 95.00% (4,080,218,880 bytes) of heap memory
Scaling row group sizes to 89.41% for 34 writers
26/08/20 21:44:32 WARN MemoryManager: Total allocation exceeds 95.00% (4,080,218,880 bytes) of heap memory
Scaling row group sizes to 86.86% for 35 writers
26/08/20 21:44:32 WARN MemoryManager: Total allocation exceeds 95.00% (4,080,218,880 bytes) of heap memory
Scaling row group sizes to 84.44% for 36 writers
26/08/20 21:44:32 WARN MemoryManager: Total allocation exceeds 9

**¿Cuántos archivos quedaron?** `.rdd.getNumPartitions()` dice cuántas particiones tiene el DataFrame — y por lo tanto, cuántos `part-0000X-...` escribe:

In [17]:
df_customers_muestra.rdd.getNumPartitions()

50

`df_customers_muestra` hereda el número de particiones de la lectura original de `customers.csv` (no se reduce solo porque `.sample()` deje menos filas) — por eso salen varias decenas de `part-0000X-...` para una muestra que en realidad es chica.

**Para compartir un solo archivo** (por ejemplo con estudiantes cuya laptop tiene pocos recursos, en vez de una carpeta con ~20 partes): junta las particiones en una sola con `.coalesce(1)` antes de escribir:

In [18]:
df_customers_muestra.coalesce(1).write.mode("overwrite").csv(f"{ARTIFACTS}/customers_muestra_csv_unico", header=True)
df_customers_muestra.coalesce(1).write.mode("overwrite").parquet(f"{ARTIFACTS}/customers_muestra_parquet_unico")

Esto sigue creando una carpeta (con un único `part-00000-...` adentro, más `_SUCCESS`) — el archivo real a compartir es ese `part-00000-...`; puedes renombrarlo o descargarlo directo desde el explorador de archivos de Jupyter.

**¿Y si quiero un `.csv` de verdad, con nombre exacto, sin carpeta?** Spark no lo hace — ni siquiera `.coalesce(1)` cambia eso (sigue generando `part-00000-...` dentro de una carpeta). Como la muestra ya es chica (~1%, cabe en memoria), conviértela a pandas y usa su `.to_csv()`, que sí escribe un único archivo con el nombre exacto:

In [19]:
df_customers_muestra.toPandas().to_csv(f"{ARTIFACTS}/customers_muestra.csv", index=False)

/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


`index=False` evita que pandas agregue una columna extra con el número de fila. Esta ruta solo es segura para datos que caben en memoria del driver — para el dataset completo (millones de filas) seguiría siendo `.coalesce(1)` + `.write.csv()` de Spark, no `.toPandas()`.

**Leer de vuelta, tenga uno o varios archivos:** Spark lee la carpeta completa como un solo DataFrame — no importa si adentro hay un `part-00000` o veinte, la lectura es igual de simple:

In [20]:
df_leido_csv = spark.read.csv(f"{ARTIFACTS}/customers_muestra_csv", header=True, inferSchema=True)
df_leido_parquet = spark.read.parquet(f"{ARTIFACTS}/customers_muestra_parquet")

df_leido_csv.count(), df_leido_parquet.count()

(13693, 13693)

## 3.5 Aplicar transformaciones y verificar la evaluación perezosa

**Producto del paso:** evidencia de que el plan se construye antes de ejecutarse.

Primero, `.select()` — elige columnas específicas del DataFrame:

In [21]:
df_seleccionado = df_customers.select("customer_id", "age", "club_member_status", "fashion_news_frequency")

Ahora, `.filter()` — conserva solo las filas que cumplen una condición:

In [22]:
from pyspark.sql.functions import col

df_activos = df_seleccionado.filter(col("club_member_status") == "ACTIVE")

Como práctica, combina ambas en una sola expresión encadenada:

In [23]:
df_activos = (
    df_customers
    .select("customer_id", "age", "club_member_status", "fashion_news_frequency")
    .filter(col("club_member_status") == "ACTIVE")
)

# Hasta aquí Spark solo construyó el plan: no hay salida, no hubo ejecución
df_activos

DataFrame[customer_id: string, age: int, club_member_status: string, fashion_news_frequency: string]

Acción: aquí recién Spark ejecuta.

In [24]:
df_activos.show(10, truncate=False)
df_activos.count()

+----------------------------------------------------------------+---+------------------+----------------------+
|customer_id                                                     |age|club_member_status|fashion_news_frequency|
+----------------------------------------------------------------+---+------------------+----------------------+
|00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657|49 |ACTIVE            |NONE                  |
|0000423b00ade91418cceaf3b26c6af3dd342b51fd051eec9c12fb36984420fa|25 |ACTIVE            |NONE                  |
|000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318|24 |ACTIVE            |NONE                  |
|00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2c5feb1ca5dff07c43e|54 |ACTIVE            |NONE                  |
|00006413d8573cd20ed7128e53b7b13819fe5cfc2d801fe7fc0f26dd8d65a85a|52 |ACTIVE            |Regularly             |
|0000757967448a6cb83efb3ea7a3fb9d418ac7adf2379d8cd0c725276a467a2a|20 |ACTIVE            |NONE   

1272491

## 3.6 Analizar el plan de ejecución con `explain()`

**Producto del paso:** plan de ejecución interpretado con al menos una optimización identificada.

In [25]:
df_activos.explain(True)

== Parsed Logical Plan ==
'Filter '`=`('club_member_status, ACTIVE)
+- Project [customer_id#597, age#602, club_member_status#600, fashion_news_frequency#601]
   +- Relation [customer_id#597,FN#598,Active#599,club_member_status#600,fashion_news_frequency#601,age#602,postal_code#603] csv

== Analyzed Logical Plan ==
customer_id: string, age: int, club_member_status: string, fashion_news_frequency: string
Filter (club_member_status#600 = ACTIVE)
+- Project [customer_id#597, age#602, club_member_status#600, fashion_news_frequency#601]
   +- Relation [customer_id#597,FN#598,Active#599,club_member_status#600,fashion_news_frequency#601,age#602,postal_code#603] csv

== Optimized Logical Plan ==
Project [customer_id#597, age#602, club_member_status#600, fashion_news_frequency#601]
+- Filter (isnotnull(club_member_status#600) AND (club_member_status#600 = ACTIVE))
   +- Relation [customer_id#597,FN#598,Active#599,club_member_status#600,fashion_news_frequency#601,age#602,postal_code#603] csv

== 

## 3.7 Aplicar funciones y crear columnas con `withColumn()`

**Producto del paso:** `df_articles` (cargado en 3.3) con al menos tres columnas nuevas o corregidas.

In [26]:
from pyspark.sql.functions import col, when, lit, current_date

**Corregir el tipo de una columna** (`.cast()`): `article_id` son solo dígitos, `inferSchema` corre el riesgo de quitarle el cero inicial.

In [27]:
df_articles = df_articles.withColumn("article_id", col("article_id").cast("string"))

**Clasificar con `when()`/`otherwise()`** (valores reales de `perceived_colour_value_name`: `"Dark"`, `"Light"`, u otros):

In [28]:
df_articles = df_articles.withColumn(
    "rango_percibido",
    when(col("perceived_colour_value_name") == "Dark", "oscuro")
    .when(col("perceived_colour_value_name") == "Light", "claro")
    .otherwise("medio")
)

**Agregar una columna constante** (`lit()`):

In [29]:
df_articles = df_articles.withColumn("fuente", lit("H&M Kaggle"))

**Agregar una columna con fecha actual** (`current_date()` — la calcula Spark, no tú):

In [30]:
df_articles = df_articles.withColumn("fecha_procesado", current_date())

Verifica el resultado — confirma que `article_id` conserva el cero inicial (ej. `0108775015`, no `108775015`):

In [31]:
df_articles.select(
    "article_id", "prod_name", "perceived_colour_value_name", "rango_percibido", "fuente", "fecha_procesado"
).show(5, truncate=False)

+----------+-----------------+---------------------------+---------------+----------+---------------+
|article_id|prod_name        |perceived_colour_value_name|rango_percibido|fuente    |fecha_procesado|
+----------+-----------------+---------------------------+---------------+----------+---------------+
|108775015 |Strap top        |Dark                       |oscuro         |H&M Kaggle|2026-08-20     |
|108775044 |Strap top        |Light                      |claro          |H&M Kaggle|2026-08-20     |
|108775051 |Strap top (1)    |Dusty Light                |medio          |H&M Kaggle|2026-08-20     |
|110065001 |OP T-shirt (Idro)|Dark                       |oscuro         |H&M Kaggle|2026-08-20     |
|110065002 |OP T-shirt (Idro)|Light                      |claro          |H&M Kaggle|2026-08-20     |
+----------+-----------------+---------------------------+---------------+----------+---------------+
only showing top 5 rows


## 3.8 Cargar `transactions.parquet` y aplicar funciones

**Producto del paso:** DataFrame `df_transactions` cargado (tercera forma de lectura: Parquet), con tipos corregidos y columnas clasificadas para poder agrupar en 3.9.

Primero, la lectura — trabajamos solo con un subconjunto (`.limit(100000)`) del archivo completo, para explorar sin procesar todo el volumen disponible:

In [32]:
df_transactions = spark.read.parquet(f"{ORIGEN_DATOS}/transactions.parquet").limit(100000)

Confirma tú mismo las columnas — no asumas el esquema:

**Fin de semana vs. laborable** (`dayofweek()` + `.isin()` — 1=domingo...7=sábado, convención propia de Spark, no ISO):

In [33]:
from pyspark.sql.functions import dayofweek

df_transactions = df_transactions.withColumn("dia_semana", dayofweek(col("t_dat")))

df_transactions = df_transactions.withColumn(
    "tipo_dia",
    when(col("dia_semana").isin(1, 7), "Fin de semana").otherwise("Laborable")
)

**Por qué no `date_format(col, "u")`:** una versión anterior de este notebook usaba `date_format(col("t_dat"), "u")`. Es un patrón de texto, y desde Spark 3.0 el parser de fechas cambió de `SimpleDateFormat` a `DateTimeFormatter` — en el parser nuevo, `"u"` significa **año**, no día de la semana; en el viejo significaba día de la semana. Como el significado cambió entre versiones, Spark no lo ejecuta en silencio: lanza `SparkUpgradeException` para forzar una decisión explícita. `dayofweek()` evita el problema porque no depende de un patrón de texto ambiguo.

**Por qué el error apareció en la celda de la función ventana (3.9) y no aquí:** evaluación perezosa. Esta celda solo arma el plan (`withColumn()` es transformación, no acción) — el error se dispara recién en la primera `.show()`/`.count()` que fuerza a evaluar toda la cadena de `df_transactions`, sin importar en qué celda esté esa acción.

In [34]:
from pyspark.sql.functions import col, to_date

df_transactions = (
    df_transactions
    .withColumn("t_dat", to_date(col("t_dat"), "yyyy-MM-dd"))
    .withColumn("customer_id", col("customer_id").cast("string"))
    .withColumn("article_id", col("article_id").cast("string"))
    .withColumn("price", col("price").cast("double"))
    .withColumn("sales_channel_id", col("sales_channel_id").cast("int"))
)

**Clasificar por canal de venta:**

In [35]:
from pyspark.sql.functions import when

df_transactions = df_transactions.withColumn(
    "canal",
    when(col("sales_channel_id") == 1, "Online")
    .when(col("sales_channel_id") == 2, "Tienda")
    .otherwise("Desconocido")
)

**Etiquetar transacciones baratas o caras:** recuerda que `price` está normalizado a [0, 1] — los umbrales van en esa escala, no en soles/dólares:

In [36]:
df_transactions = df_transactions.withColumn(
    "categoria_precio",
    when(col("price") < 0.1, "Barato")
    .when(col("price") < 0.3, "Medio")
    .otherwise("Caro")
)

**Aplicar múltiples condiciones** (con `&`, combinando dos columnas):

In [37]:
df_transactions = df_transactions.withColumn(
    "tipo_transaccion",
    when((col("sales_channel_id") == 1) & (col("price") > 0.3), "Online Premium")
    .when((col("sales_channel_id") == 2) & (col("price") > 0.3), "Tienda Premium")
    .otherwise("Regular")
)

**Fin de semana vs. laborable** (`date_format()` + `.isin()` — `"u"` es el día ISO de la semana, 1=lunes...7=domingo, no confundir con `"d"` que es día del mes):

In [38]:
from pyspark.sql.functions import date_format

df_transactions = df_transactions.withColumn("dia_semana", date_format(col("t_dat"), "u"))

df_transactions = df_transactions.withColumn(
    "tipo_dia",
    when(col("dia_semana").isin("6", "7"), "Fin de semana").otherwise("Laborable")
)

## 3.9 Aplicar agrupaciones y agregaciones (`transactions.parquet`)

**Producto del paso:** resumen agregado por cliente, con la advertencia de dominio sobre `price` aplicada.

**Total gastado por cliente** (`sum`):

In [39]:
from pyspark.sql.functions import sum

df_total_por_cliente = df_transactions.groupBy("customer_id").agg(
    sum("price").alias("total_normalizado")
)
df_total_por_cliente.show(5)

+--------------------+------------------+
|         customer_id| total_normalizado|
+--------------------+------------------+
|000058a12d5b43e67...| 0.081322033898305|
|00007d2de826758b6...|0.0863559322033897|
|00083cda041544b2f...| 0.190593220338983|
|0008968c0d451dbc5...|0.0428474576271185|
|000aa7f0dc06cd717...|0.7130508474576257|
+--------------------+------------------+
only showing top 5 rows


**Promedio de gasto por cliente** (`avg`):

In [40]:
from pyspark.sql.functions import avg

df_avg_por_cliente = df_transactions.groupBy("customer_id").agg(
    avg("price").alias("promedio_normalizado")
)
df_avg_por_cliente.show(5)

+--------------------+--------------------+
|         customer_id|promedio_normalizado|
+--------------------+--------------------+
|000058a12d5b43e67...|  0.0406610169491525|
|00007d2de826758b6...| 0.01727118644067794|
|00083cda041544b2f...|0.038118644067796595|
|0008968c0d451dbc5...| 0.02142372881355925|
|000aa7f0dc06cd717...|0.023768361581920857|
+--------------------+--------------------+
only showing top 5 rows


**Número de transacciones por cliente** (`count`):

In [41]:
from pyspark.sql.functions import count

df_count_por_cliente = df_transactions.groupBy("customer_id").agg(
    count("*").alias("num_transacciones")
)
df_count_por_cliente.show(5)

+--------------------+-----------------+
|         customer_id|num_transacciones|
+--------------------+-----------------+
|000058a12d5b43e67...|                2|
|00007d2de826758b6...|                5|
|00083cda041544b2f...|                5|
|0008968c0d451dbc5...|                2|
|000aa7f0dc06cd717...|               30|
+--------------------+-----------------+
only showing top 5 rows


**Clientes únicos por día** (`countDistinct`):

In [42]:
from pyspark.sql.functions import countDistinct

df_clientes_unicos_por_dia = df_transactions.groupBy("t_dat").agg(
    countDistinct("customer_id").alias("clientes_unicos")
)
df_clientes_unicos_por_dia.show(5)

+----------+---------------+
|     t_dat|clientes_unicos|
+----------+---------------+
|2018-09-20|          13987|
|2018-09-21|          13932|
|2018-09-22|           1285|
+----------+---------------+



**Varias agregaciones a la vez** (más eficiente que calcularlas por separado):

In [43]:
from pyspark.sql.functions import sum, count, avg

df_agg_multi = df_transactions.groupBy("customer_id").agg(
    count("*").alias("num_transacciones"),
    sum("price").alias("total_normalizado"),
    avg("price").alias("promedio_normalizado")
)
df_agg_multi.show(5)

+--------------------+-----------------+------------------+--------------------+
|         customer_id|num_transacciones| total_normalizado|promedio_normalizado|
+--------------------+-----------------+------------------+--------------------+
|000058a12d5b43e67...|                2| 0.081322033898305|  0.0406610169491525|
|00007d2de826758b6...|                5|0.0863559322033897| 0.01727118644067794|
|00083cda041544b2f...|                5| 0.190593220338983|0.038118644067796595|
|0008968c0d451dbc5...|                2|0.0428474576271185| 0.02142372881355925|
|000aa7f0dc06cd717...|               30|0.7130508474576257|0.023768361581920857|
+--------------------+-----------------+------------------+--------------------+
only showing top 5 rows


**Agrupar por múltiples columnas:**

In [44]:
df_ventas_por_dia_y_canal = df_transactions.groupBy("t_dat", "sales_channel_id").agg(
    sum("price").alias("total_normalizado")
)
df_ventas_por_dia_y_canal.show(5)

+----------+----------------+------------------+
|     t_dat|sales_channel_id| total_normalizado|
+----------+----------------+------------------+
|2018-09-20|               2| 1072.852593220397|
|2018-09-20|               1|342.49325423728436|
|2018-09-21|               2|1036.5931525424558|
|2018-09-21|               1| 382.6691525423702|
|2018-09-22|               1|55.550881355931914|
+----------+----------------+------------------+
only showing top 5 rows


**Usar `agg()` con un diccionario** (forma alternativa, sin `alias()`):

In [45]:
df_agg_dict = (
    df_transactions.groupBy("customer_id")
    .agg({"price": "sum", "article_id": "count"})
    .withColumnRenamed("sum(price)", "total_normalizado")
    .withColumnRenamed("count(article_id)", "num_articulos")
)
df_agg_dict.show(5)

+--------------------+-------------+------------------+
|         customer_id|num_articulos| total_normalizado|
+--------------------+-------------+------------------+
|000058a12d5b43e67...|            2| 0.081322033898305|
|00007d2de826758b6...|            5|0.0863559322033897|
|00083cda041544b2f...|            5| 0.190593220338983|
|0008968c0d451dbc5...|            2|0.0428474576271185|
|000aa7f0dc06cd717...|           30|0.7130508474576257|
+--------------------+-------------+------------------+
only showing top 5 rows


**Total gastado por cliente, sin agrupar** (función ventana — a diferencia de `groupBy().agg()`, no colapsa filas):

In [46]:
from pyspark.sql.window import Window

window_cliente = Window.partitionBy("customer_id")

In [47]:
from pyspark.sql.functions import sum

df_con_total_cliente = df_transactions.withColumn(
    "total_gastado_cliente",
    sum("price").over(window_cliente)
)
df_con_total_cliente.show(5)

SparkUpgradeException: [INCONSISTENT_BEHAVIOR_CROSS_VERSION.DATETIME_PATTERN_RECOGNITION] You may get a different result due to the upgrading to Spark >= 3.0:
Fail to recognize 'u' pattern in the DateTimeFormatter.
1) You can set "spark.sql.legacy.timeParserPolicy" to "LEGACY" to restore the behavior before Spark 3.0.
2) You can form a valid datetime pattern with the guide from 'https://spark.apache.org/docs/latest/sql-ref-datetime-pattern.html'. SQLSTATE: 42K0B

**Advertencia de dominio:** `price` está normalizado por Kaggle a [0, 1] — no representa una moneda real. `sum("price")`/`avg("price")` son agregaciones técnicamente correctas, pero leerlas como "gasto en soles/dólares" sería un error de dominio.

## 3.10 Convertir a RDD y procesar texto (`detail_desc` de `articles.csv`)

**Producto del paso:** conteo de palabras distribuido sobre descripciones de producto, con las 10 más frecuentes.

In [ ]:
import re
from operator import add

rdd = df_articles.select("detail_desc").rdd.map(lambda x: x.detail_desc)
rdd = rdd.filter(lambda texto: texto is not None)  # algunos artículos no tienen descripción

palabras = rdd.flatMap(
    lambda linea: re.sub(r"[^\wáéíóúñüÁÉÍÓÚÑÜ]", " ", linea.lower()).split()
)

pares = palabras.filter(lambda p: p != "").map(lambda palabra: (palabra, 1))
conteo = pares.reduceByKey(add)

conteo.takeOrdered(10, key=lambda x: -x[1])

## 3.11 Documentar hallazgos y responder preguntas de reflexión

**Producto del paso:** notebook documentado con celdas markdown explicando cada resultado.

Agrega debajo de cada bloque anterior una breve explicación de qué hiciste y qué observaste — es la base directa de la evidencia técnica para 4.3.1.

**Reflexión técnica breve** (5 a 8 líneas): ¿por qué la evaluación perezosa es útil para procesar datos a escala, y qué riesgo tendría si Spark ejecutara cada transformación de inmediato, apenas se escribe?

_(Responde aquí)_